In [8]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd().parent.parent.parent

sys.path.insert(0, str(ROOT))
ROOT


PosixPath('/home/sakana/Code/ManifoldsTeam/woodbench')

# Task B — đối chiếu F1 cũ và F1 hiện tại

Notebook này tạo bảng có **đúng hai cột F1**:

1. `F1_cu_20_output_19_test_reported`: F1 đã lưu trong artifact strict cũ. Prediction cũ không còn nên cột này **không thể tái tính**.
2. `F1_moi_17_lop_recomputed`: F1 tính lại từ `y_true.npy`/`y_pred.npy` mới nhất bằng `src.eval.metrics.compute_task_b_metrics`.

Các cột định danh `(experiment, seed, backbone)` được đặt làm index, không tính là cột metric. Delta và chẩn đoán các dòng lệch quá 0,5 điểm phần trăm được xuất thành bảng riêng.

> Hai cột thuộc hai benchmark khác nhau về class universe, test set và checkpoint. Vì vậy delta dùng để kiểm toán lịch sử, không phải phép đo thuần túy tác động của việc sửa công thức metric.


In [9]:
import inspect
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.eval.metrics import (
    TASK_B_LABELS,
    TASK_B_NUM_CLASSES,
    compute_task_b_metrics,
)

REQUESTED_ARCHIVE_ROOT = ROOT / 'data' / 'experiments-old' / 'task-B'
LEGACY_STRICT_ROOT = ROOT / 'data' / 'experiments' / 'Task-B [Deprecated]'
CURRENT_EVAL_ROOT = (
    ROOT / 'data' / 'experiments' / 'phase_1b' / 'best_checkpoint_evaluation'
)
CURRENT_DATA_ROOT = ROOT / 'data' / 'task-B-group-split'
LEGACY_EXCLUDED_ROOT = ROOT / 'data' / 'task-B-group-split-excluded'
OUTPUT_DIR = ROOT / 'data' / 'experiments' / 'phase_1b' / 'metric_audit'
THRESHOLD_PP = 0.5
EXPECTED_RUNS = 30

assert TASK_B_NUM_CLASSES == 17
assert tuple(TASK_B_LABELS) == tuple(range(17))


## 1. Kiểm kê và chọn đúng artifact

Trong workspace hiện tại, `experiments-old/task-B` chứa bộ original-split cũ hơn và có per-class metric đủ 20 lớp. Bộ khớp mô tả strict **20 output nhưng test có 19 lớp** nằm tại `experiments/Task-B [Deprecated]`. Notebook kiểm kê cả hai nhưng chỉ dùng bộ strict để đối chiếu.


In [10]:
def count_files(root, pattern):
    return len(list(root.rglob(pattern))) if root.exists() else 0


inventory = pd.DataFrame(
    [
        {
            'artifact': 'archive được nhắc tới',
            'root': str(REQUESTED_ARCHIVE_ROOT.relative_to(ROOT)),
            'test_metrics_json': count_files(REQUESTED_ARCHIVE_ROOT, 'test_metrics.json'),
            'y_true_npy': count_files(REQUESTED_ARCHIVE_ROOT, 'y_true.npy'),
            'y_pred_npy': count_files(REQUESTED_ARCHIVE_ROOT, 'y_pred.npy'),
            'selected': False,
            'reason': 'original split; không khớp strict test 19 lớp',
        },
        {
            'artifact': 'strict cũ',
            'root': str(LEGACY_STRICT_ROOT.relative_to(ROOT)),
            'test_metrics_json': count_files(LEGACY_STRICT_ROOT, 'test_metrics.json'),
            'y_true_npy': count_files(LEGACY_STRICT_ROOT, 'y_true.npy'),
            'y_pred_npy': count_files(LEGACY_STRICT_ROOT, 'y_pred.npy'),
            'selected': True,
            'reason': '20 output; strict test được ghi nhận có 19 lớp',
        },
        {
            'artifact': 'Final-B mới nhất',
            'root': str(CURRENT_EVAL_ROOT.relative_to(ROOT)),
            'test_metrics_json': count_files(CURRENT_EVAL_ROOT, 'metrics.json'),
            'y_true_npy': count_files(CURRENT_EVAL_ROOT, 'y_true.npy'),
            'y_pred_npy': count_files(CURRENT_EVAL_ROOT, 'y_pred.npy'),
            'selected': True,
            'reason': 'prediction bundle của best checkpoint, 17 lớp',
        },
    ]
)
display(inventory)

assert LEGACY_STRICT_ROOT.exists(), LEGACY_STRICT_ROOT
assert CURRENT_EVAL_ROOT.exists(), CURRENT_EVAL_ROOT
assert count_files(LEGACY_STRICT_ROOT, 'test_metrics.json') == EXPECTED_RUNS
assert count_files(LEGACY_STRICT_ROOT, 'y_true.npy') == 0
assert count_files(LEGACY_STRICT_ROOT, 'y_pred.npy') == 0
assert count_files(CURRENT_EVAL_ROOT, 'y_true.npy') == EXPECTED_RUNS
assert count_files(CURRENT_EVAL_ROOT, 'y_pred.npy') == EXPECTED_RUNS


,artifact,root,test_metrics_json,y_true_npy,y_pred_npy,selected,reason
0,archive được nhắc tới,data/experiments-old/task-B,30,0,0,False,original split; không khớp strict test 19 lớp
1,strict cũ,data/experiments/Task-B [Deprecated],30,0,0,True,20 output; strict test được ghi nhận có 19 lớp
2,Final-B mới nhất,data/experiments/phase_1b/best_checkpoint_eval...,30,30,30,True,"prediction bundle của best checkpoint, 17 lớp"


In [11]:
def folder_classes(root, split):
    split_dir = root / split
    if not split_dir.exists():
        return set()
    return {path.name for path in split_dir.iterdir() if path.is_dir()}


coverage_rows = []
for split in ['train', 'val', 'test']:
    final_classes = folder_classes(CURRENT_DATA_ROOT, split)
    excluded_classes = folder_classes(LEGACY_EXCLUDED_ROOT, split)
    legacy_classes = final_classes | excluded_classes
    coverage_rows.append(
        {
            'split': split,
            'final_b_classes': len(final_classes),
            'legacy_excluded_classes': len(excluded_classes),
            'reconstructed_legacy_classes': len(legacy_classes),
            'missing_from_legacy_test': (
                'Calocedrus sp' if split == 'test' else ''
            ),
        }
    )

coverage = pd.DataFrame(coverage_rows)
display(coverage)
assert coverage.set_index('split').loc['train', 'reconstructed_legacy_classes'] == 20
assert coverage.set_index('split').loc['test', 'reconstructed_legacy_classes'] == 19
assert coverage.set_index('split').loc['test', 'final_b_classes'] == 17


,split,final_b_classes,legacy_excluded_classes,reconstructed_legacy_classes,missing_from_legacy_test
0,train,17,3,20,
1,val,17,0,17,
2,test,17,2,19,Calocedrus sp


## 2. Đọc F1 cũ và tính lại F1 mới

F1 cũ được đọc nguyên trạng từ `test_metrics.json`. Notebook không gọi nó là “recomputed” vì không còn prediction arrays. F1 mới luôn được tính lại từ prediction bundle bằng contract Task B 17 lớp dùng chung.


In [12]:
def experiment_from_legacy_path(path):
    first = path.relative_to(LEGACY_STRICT_ROOT).parts[0]
    return first


def load_legacy_reported_f1():
    rows = []
    for path in sorted(LEGACY_STRICT_ROOT.rglob('test_metrics.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        f1 = payload.get('test_f1_macro', payload.get('f1_macro'))
        if f1 is None:
            raise KeyError(f'Không có macro-F1 trong {path}')
        rows.append(
            {
                'experiment': payload.get(
                    'experiment', experiment_from_legacy_path(path)
                ),
                'seed': int(payload.get('seed', 42)),
                'backbone': payload.get('backbone', path.parent.name),
                'F1_cu_20_output_19_test_reported': 100.0 * float(f1),
                'legacy_metric_path': str(path.relative_to(ROOT)),
            }
        )
    return pd.DataFrame(rows)


def load_current_recomputed_f1():
    rows = []
    for path in sorted(CURRENT_EVAL_ROOT.rglob('metrics.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        run_dir = path.parent
        y_true = np.load(run_dir / 'y_true.npy')
        y_pred = np.load(run_dir / 'y_pred.npy')
        class_names = json.loads(
            (run_dir / 'class_names.json').read_text(encoding='utf-8')
        )
        recalculated = compute_task_b_metrics(
            y_true, y_pred, class_names=class_names
        )
        assert len(class_names) == TASK_B_NUM_CLASSES
        assert sorted(np.unique(y_true).tolist()) == list(TASK_B_LABELS)
        assert np.isclose(recalculated['f1_macro'], payload['f1_macro'])
        rows.append(
            {
                'experiment': payload['experiment'],
                'seed': int(payload['seed']),
                'backbone': payload['backbone'],
                'F1_moi_17_lop_recomputed': 100.0 * recalculated['f1_macro'],
                'current_metric_path': str(path.relative_to(ROOT)),
            }
        )
    return pd.DataFrame(rows)


legacy_df = load_legacy_reported_f1()
current_df = load_current_recomputed_f1()
keys = ['experiment', 'seed', 'backbone']

assert len(legacy_df) == EXPECTED_RUNS
assert len(current_df) == EXPECTED_RUNS
assert not legacy_df.duplicated(keys).any()
assert not current_df.duplicated(keys).any()
assert set(map(tuple, legacy_df[keys].to_numpy())) == set(
    map(tuple, current_df[keys].to_numpy())
)

audit = legacy_df.merge(current_df, on=keys, validate='one_to_one')
audit['delta_pp'] = (
    audit['F1_moi_17_lop_recomputed']
    - audit['F1_cu_20_output_19_test_reported']
)
audit['abs_delta_pp'] = audit['delta_pp'].abs()
audit['lech_qua_0_5_pp'] = audit['abs_delta_pp'] > THRESHOLD_PP
audit = audit.sort_values(keys).reset_index(drop=True)
print(f'Đã ghép one-to-one {len(audit)} run.')


Đã ghép one-to-one 30 run.


## 3. Bảng F1 hai cột


In [13]:
f1_columns = [
    'F1_cu_20_output_19_test_reported',
    'F1_moi_17_lop_recomputed',
]
comparison_table = audit.set_index(keys)[f1_columns]
display(comparison_table.style.format('{:.3f}'))

main_summary = (
    audit[audit['experiment'] == 'main']
    .groupby('backbone', sort=True)[f1_columns]
    .mean()
)
display(Markdown('### Bảng tóm tắt main — trung bình ba seed'))
display(main_summary.style.format('{:.3f}'))


### Bảng tóm tắt main — trung bình ba seed

,F1_cu_20_output_19_test_reported,F1_moi_17_lop_recomputed
backbone,,
convnext_tiny,98.895,99.524
efficientnet_b3,92.143,97.138
mobilenetv3_large,89.031,98.407
resnet50,82.211,93.844
swin_tiny,99.411,99.481


## 4. Chẩn đoán các dòng lệch quá 0,5 điểm phần trăm

Bảng dưới chỉ ra vị trí code làm **khác định nghĩa metric**. Nó không tuyên bố dòng code đó gây ra toàn bộ delta, vì hai cột còn khác dataset, số lớp và checkpoint. Không có `y_true/y_pred` cũ nên không thể tách định lượng từng nguyên nhân.


In [14]:
LEGACY_NOTEBOOK_ROOT = (
    ROOT / 'notebooks' / 'phase 1' / 'task-B [DEPRECATED]'
)


def legacy_notebook_path(experiment, backbone):
    if experiment == 'main':
        return LEGACY_NOTEBOOK_ROOT / 'M1_main' / f'TaskB_{backbone}.ipynb'
    if experiment == 'taskB_90rotation':
        return (
            LEGACY_NOTEBOOK_ROOT
            / 'M2_ablation'
            / '3.1_90rotation'
            / f'TaskB_90rot_{backbone}.ipynb'
        )
    if experiment in {
        'taskB_label_smoothing_0.0',
        'taskB_label_smoothing_0.2',
    }:
        return (
            LEGACY_NOTEBOOK_ROOT
            / 'M2_ablation'
            / '3.2_label_smoothing'
            / f'TaskB_ls_{backbone}.ipynb'
        )
    raise ValueError(f'Experiment không được hỗ trợ: {experiment}')


def find_legacy_f1_call(experiment, backbone):
    path = legacy_notebook_path(experiment, backbone)
    notebook = json.loads(path.read_text(encoding='utf-8'))
    for cell_number, cell in enumerate(notebook['cells'], start=1):
        if cell.get('cell_type') != 'code':
            continue
        source_lines = cell.get('source', [])
        for line_number, line in enumerate(source_lines, start=1):
            if 'f1_score(' in line:
                location = (
                    f'{path.relative_to(ROOT)}: cell {cell_number}, '
                    f'line {line_number}'
                )
                expression_lines = []
                for expression_line in source_lines[line_number - 1:]:
                    expression_lines.append(expression_line.strip())
                    if ')' in expression_line:
                        break
                expression = ' '.join(expression_lines)
                if 'average="macro"' in expression:
                    return location, expression
    raise ValueError(f'Không tìm thấy lời gọi F1 trong {path}')


metric_source = Path(inspect.getsourcefile(compute_task_b_metrics)).resolve()
_, metric_start_line = inspect.getsourcelines(compute_task_b_metrics)
current_location = (
    f'{metric_source.relative_to(ROOT)}:{metric_start_line}; '
    'labels cố định tại src/eval/metrics.py:21-22'
)

source_rows = []
for experiment, backbone in sorted(
    set(map(tuple, audit[['experiment', 'backbone']].to_numpy()))
):
    legacy_location, legacy_expression = find_legacy_f1_call(
        experiment, backbone
    )
    source_rows.append(
        {
            'experiment': experiment,
            'backbone': backbone,
            'legacy_code_location': legacy_location,
            'legacy_expression': legacy_expression,
        }
    )
source_map = pd.DataFrame(source_rows)

flagged = audit[audit['lech_qua_0_5_pp']].merge(
    source_map, on=['experiment', 'backbone'], validate='many_to_one'
)
flagged['current_code_location'] = current_location
flagged['ket_luan_nguyen_nhan'] = (
    'Legacy gọi f1_score(..., average="macro", zero_division=0) '
    'nhưng không truyền labels; code mới cố định 17 labels. Không thể quy '
    'toàn bộ delta cho dòng này vì dataset/class universe/checkpoint đã đổi '
    'và prediction cũ không còn.'
)
diagnostic_columns = [
    *keys,
    'delta_pp',
    'legacy_code_location',
    'legacy_expression',
    'current_code_location',
    'ket_luan_nguyen_nhan',
]
display(flagged[diagnostic_columns].style.format({'delta_pp': '{:+.3f}'}))
print(f'{len(flagged)}/{len(audit)} run lệch quá {THRESHOLD_PP} pp.')


,experiment,seed,backbone,delta_pp,legacy_code_location,legacy_expression,current_code_location,ket_luan_nguyen_nhan
0,main,42,convnext_tiny,+0.672,"notebooks/phase 1/task-B [DEPRECATED]/M1_main/TaskB_convnext_tiny.ipynb: cell 7, line 59","""test_f1_macro"": f1_score(all_labels, all_preds, average=""macro"", zero_division=0),",src/eval/metrics.py:125; labels cố định tại src/eval/metrics.py:21-22,"Legacy gọi f1_score(..., average=""macro"", zero_division=0) nhưng không truyền labels; code mới cố định 17 labels. Không thể quy toàn bộ delta cho dòng này vì dataset/class universe/checkpoint đã đổi và prediction cũ không còn."
1,main,42,efficientnet_b3,+4.269,"notebooks/phase 1/task-B [DEPRECATED]/M1_main/TaskB_efficientnet_b3.ipynb: cell 7, line 59","""test_f1_macro"": f1_score(all_labels, all_preds, average=""macro"", zero_division=0),",src/eval/metrics.py:125; labels cố định tại src/eval/metrics.py:21-22,"Legacy gọi f1_score(..., average=""macro"", zero_division=0) nhưng không truyền labels; code mới cố định 17 labels. Không thể quy toàn bộ delta cho dòng này vì dataset/class universe/checkpoint đã đổi và prediction cũ không còn."
2,main,42,mobilenetv3_large,+9.638,"notebooks/phase 1/task-B [DEPRECATED]/M1_main/TaskB_mobilenetv3_large.ipynb: cell 7, line 59","""test_f1_macro"": f1_score(all_labels, all_preds, average=""macro"", zero_division=0),",src/eval/metrics.py:125; labels cố định tại src/eval/metrics.py:21-22,"Legacy gọi f1_score(..., average=""macro"", zero_division=0) nhưng không truyền labels; code mới cố định 17 labels. Không thể quy toàn bộ delta cho dòng này vì dataset/class universe/checkpoint đã đổi và prediction cũ không còn."
3,main,42,resnet50,+12.983,"notebooks/phase 1/task-B [DEPRECATED]/M1_main/TaskB_resnet50.ipynb: cell 7, line 59","""test_f1_macro"": f1_score(all_labels, all_preds, average=""macro"", zero_division=0),",src/eval/metrics.py:125; labels cố định tại src/eval/metrics.py:21-22,"Legacy gọi f1_score(..., average=""macro"", zero_division=0) nhưng không truyền labels; code mới cố định 17 labels. Không thể quy toàn bộ delta cho dòng này vì dataset/class universe/checkpoint đã đổi và prediction cũ không còn."
4,main,123,efficientnet_b3,+5.272,"notebooks/phase 1/task-B [DEPRECATED]/M1_main/TaskB_efficientnet_b3.ipynb: cell 7, line 59","""test_f1_macro"": f1_score(all_labels, all_preds, average=""macro"", zero_division=0),",src/eval/metrics.py:125; labels cố định tại src/eval/metrics.py:21-22,"Legacy gọi f1_score(..., average=""macro"", zero_division=0) nhưng không truyền labels; code mới cố định 17 labels. Không thể quy toàn bộ delta cho dòng này vì dataset/class universe/checkpoint đã đổi và prediction cũ không còn."
5,main,123,mobilenetv3_large,+10.359,"notebooks/phase 1/task-B [DEPRECATED]/M1_main/TaskB_mobilenetv3_large.ipynb: cell 7, line 59","""test_f1_macro"": f1_score(all_labels, all_preds, average=""macro"", zero_division=0),",src/eval/metrics.py:125; labels cố định tại src/eval/metrics.py:21-22,"Legacy gọi f1_score(..., average=""macro"", zero_division=0) nhưng không truyền labels; code mới cố định 17 labels. Không thể quy toàn bộ delta cho dòng này vì dataset/class universe/checkpoint đã đổi và prediction cũ không còn."
6,main,123,resnet50,+10.275,"notebooks/phase 1/task-B [DEPRECATED]/M1_main/TaskB_resnet50.ipynb: cell 7, line 59","""test_f1_macro"": f1_score(all_labels, all_preds, average=""macro"", zero_division=0),",src/eval/metrics.py:125; labels cố định tại src/eval/metrics.py:21-22,"Legacy gọi f1_score(..., average=""macro"", zero_division=0) nhưng không truyền labels; code mới cố định 17 labels. Không thể quy toàn bộ delta cho dòng này vì dataset/class universe/checkpoint đã đổi và prediction cũ không còn."
7,main,456,convnext_tiny,+0.897,"notebooks/phase 1/task-B [DEPRECATED]/M1_main/TaskB_convnext_tiny.ipynb: cell 7, line 59","""test_f1_macro"": f1_score(all_labels, all_preds, average=""macro"", zero_division=0),",src/eval/metrics.py:125; labels cố định tại src/eval/metrics.py:21-22

20/30 run lệch quá 0.5 pp.


## 5. Xuất file phục vụ báo cáo


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison_path = OUTPUT_DIR / 'task_b_f1_old_vs_current_two_columns.csv'
summary_path = OUTPUT_DIR / 'task_b_f1_old_vs_current_main_summary.csv'
diagnostics_path = OUTPUT_DIR / 'task_b_f1_delta_over_0_5pp_diagnostics.csv'
inventory_path = OUTPUT_DIR / 'task_b_metric_artifact_inventory.csv'

comparison_table.to_csv(comparison_path, float_format='%.6f')
main_summary.to_csv(summary_path, float_format='%.6f')
flagged[diagnostic_columns].to_csv(
    diagnostics_path, index=False, float_format='%.6f'
)
inventory.to_csv(inventory_path, index=False)

for path in [comparison_path, summary_path, diagnostics_path, inventory_path]:
    print(path.relative_to(ROOT))


## Cách diễn giải trong bài báo

- Cột cũ là số **reported**, không phải số tái tính, vì artifact cũ không còn prediction arrays.
- Cột mới là số **recomputed** từ prediction bundle best-checkpoint và contract metric Task B 17 lớp.
- Không dùng delta giữa hai cột để kết luận riêng tác động của sửa metric: lần chạy mới đồng thời thay class universe, test set và checkpoint.
- Dòng code cũ thiếu `labels=` là một lỗi khả năng tái lập. Muốn đo chính xác tác động của lỗi đó cần `y_true/y_pred` cũ; checkpoint đơn lẻ không đủ nếu test set/mapping cũ không còn nguyên vẹn.
